<a href="https://colab.research.google.com/github/RashmiBansode2002/Conversion-of-Text-to-Gloss-for-Sign-Language/blob/main/tokenize_isl_custom.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#imports
import re
import pandas as pd
import csv
import re
from pathlib import Path
from tqdm import tqdm
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding, Attention

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# when putting () ->

pattern = "(?=[1-9|*|^])|(?<=[1-9|*|^])"

### Define in and out file

In [ ]:
# data_dir = Path("C:/Users/Dell/Desktop/uni/8-Semester/bachelor/dgs-corpus/")

# glosses = data_dir / "gloss-sentences-cleaned.csv"

# glosses_out = data_dir / 'glosses_tok.csv'

# assert glosses.exists(), 'glosses not found'



# Define the path to the folder containing your files on Google Drive
# data_dir = Path("/content/drive/My Drive/path_to_folder/")

# # Define the filenames for input and output files
# glosses = data_dir / "gloss-sentences-cleaned.csv"
# glosses_out = data_dir / 'glosses_tok.csv'

# # Check if the input file exists
# assert glosses.exists(), 'glosses not found'


glosses = Path("/content/drive/MyDrive/text-to-gloss-sign-language-translation-main/Data/gloss 1 train.csv")
glosses_out = Path("/content/drive/MyDrive/text-to-gloss-sign-language-translation-main/Data/gloss 3 dev.xlsx")

assert glosses.exists(), 'glosses not found'

In [ ]:
%%time
def extract_stem(in_csv, out_csv):
    df = pd.read_csv(in_csv, header=None, encoding='utf-8')
    out_lines = df[0].tolist()

    with open(out_csv, 'w', encoding='utf-8') as f:
        for line in out_lines:
            splitted = line.split(" ")
            gloss_sentence_tok = []
            for i in range(0, len(splitted)):
                gloss = re.split(pattern, splitted[i], 1)
                if (len(gloss) == 2):
                    postfix = " @@" + gloss[1]
                    tok_gloss = gloss[0] + postfix
                    if ("-" in tok_gloss):
                        gloss_sentence_tok.append(tok_gloss.replace("-", " @@-@@ "))
                    else:
                        gloss_sentence_tok.append(tok_gloss)
                else:
                    if ("-" in gloss[0]):
                        gloss_sentence_tok.append(gloss[0].replace("-", " @@-@@ "))
                    else:
                        gloss_sentence_tok.append(gloss[0])

            makeastring = " ".join(gloss_sentence_tok)
            print(makeastring)
            f.write(makeastring)
            f.write('\n')

CPU times: user 7 µs, sys: 0 ns, total: 7 µs
Wall time: 10 µs


In [ ]:
%%time
extract_stem(glosses, glosses_out)

gloss
I angry
you angry
angry you me 
you angry not
I angry very
I angry very
you angry very
promise I angry not 
you bad
i bad
i bad big
promise i try  bad not
i bad want
i sleep bad
i crying
promise i cry not
help him he crying
cry means good not
cry mean bad
cry don't
help help need same cry
help help need ask don't  
you cry why , you want what
i good
he good
he night sleep good
he good very how?
promise you future good 
your name good
 your name nice
he name good have
he good, him help 
he good very 
you need help what?
promise you help 
your help i need not
you help need?
i sleep help need
i help want 
you how?
i nice how>?
you bad very how?
your sleep good?
you run how?
you me need how ?
you help me how?
soryy i say how?
thank you how tell me 
you nice can how? 
meet i you want
we meet can how ?
promise i you meet  
i meet can't sorry very
meet you nice very
my sleep bad
your name what
your name nice
that nice not
you nice not
you nice
promise i sleep
promice
sleep how?
sorry fe

In [ ]:
import pandas as pd
import nltk
from nltk.tokenize import word_tokenize
from nltk import pos_tag

nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [ ]:
# Assuming your file is named 'your_dataset.xlsx'
df = pd.read_excel("/content/drive/MyDrive/text-to-gloss-sign-language-translation-main/Data/Copy of Sentences to Glosses.xlsx")

df.head()
df.columns
# Extract input and target texts from DataFrame
input_texts = df['text'].tolist()
target_texts = df['gloss'].tolist()

In [ ]:
def remove_stopwords(sentence):
    stop_words = set(["am", "is", "the", "an", "a", "its", "are"])
    word_tokens = word_tokenize(sentence)
    filtered_sentence = [w for w in word_tokens if not w.lower() in stop_words]
    return ' '.join(filtered_sentence)

def rearrange_sentence(sentence):
    tagged_words = pos_tag(word_tokenize(sentence))
    nouns = []
    verbs_adjectives = []
    for word, tag in tagged_words:
        if tag.startswith('NN'):
            nouns.append(word)
        else:
            verbs_adjectives.append(word)
    rearranged_sentence = ' '.join(nouns + verbs_adjectives)
    return rearranged_sentence

df = pd.read_excel('/content/drive/MyDrive/Copy of Sentences to Glosses.xlsx')

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import LSTM, Input, Dense, Embedding, Attention, Concatenate
from tensorflow.keras.models import Model

# Tokenize input and target texts
input_tokenizer = tf.keras.preprocessing.text.Tokenizer(filters='')
input_tokenizer.fit_on_texts(input_texts)
input_seq = input_tokenizer.texts_to_sequences(input_texts)
max_input_length = max(len(seq) for seq in input_seq)

target_tokenizer = tf.keras.preprocessing.text.Tokenizer(filters='')
target_tokenizer.fit_on_texts(target_texts)
target_seq = target_tokenizer.texts_to_sequences(target_texts)
max_target_length = max(len(seq) for seq in target_seq)

# Pad sequences
input_seq = tf.keras.preprocessing.sequence.pad_sequences(input_seq, maxlen=max_input_length, padding='post')
target_seq = tf.keras.preprocessing.sequence.pad_sequences(target_seq, maxlen=max_target_length, padding='post')

# Create training data
encoder_input_data = input_seq
decoder_input_data = target_seq[:, :-1]  # Exclude the last token
decoder_target_data = target_seq[:, 1:]  # Exclude the first token

# Define model architecture
embedding_dim = 64
units = 128

# Encoder
encoder_inputs = Input(shape=(max_input_length,))
encoder_embedding = Embedding(len(input_tokenizer.word_index) + 1, embedding_dim)(encoder_inputs)
encoder, encoder_state_h, encoder_state_c = LSTM(units, return_state=True)(encoder_embedding)
encoder_states = [encoder_state_h, encoder_state_c]

# Decoder
decoder_inputs = Input(shape=(max_target_length - 1,))
decoder_embedding = Embedding(len(target_tokenizer.word_index) + 1, embedding_dim)(decoder_inputs)
decoder_lstm = LSTM(units, return_sequences=True, return_state=True)

decoder_outputs, _, _ = decoder_lstm(decoder_embedding, initial_state=encoder_states)

# Attention layer
attention = Attention()([decoder_outputs, encoder])

decoder_concat = Concatenate(axis=-1)([decoder_outputs, attention])

decoder_dense = Dense(len(target_tokenizer.word_index) + 1, activation='softmax')
decoder_outputs = decoder_dense(decoder_concat)

model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

# Compile the model
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Train the model
model.fit([encoder_input_data, decoder_input_data], decoder_target_data, epochs=50, batch_size=2, validation_split=0.2)

# Save the model for later use
model.save('text_to_gloss_model1.h5')

# Inference models
encoder_model = Model(encoder_inputs, encoder_states)

decoder_state_input_h = Input(shape=(units,))
decoder_state_input_c = Input(shape=(units,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

decoder_outputs, state_h, state_c = decoder_lstm(decoder_embedding, initial_state=decoder_states_inputs)

Epoch 1/50
34/34 [==============================] - 6s 46ms/step - loss: 2.7800 - accuracy: 0.5858 - val_loss: 1.5632 - val_accuracy: 0.6961
Epoch 2/50
34/34 [==============================] - 1s 16ms/step - loss: 1.7176 - accuracy: 0.6054 - val_loss: 1.4523 - val_accuracy: 0.6961
Epoch 3/50
34/34 [==============================] - 1s 17ms/step - loss: 1.6402 - accuracy: 0.6054 - val_loss: 1.4633 - val_accuracy: 0.6863
Epoch 4/50
34/34 [==============================] - 1s 16ms/step - loss: 1.5781 - accuracy: 0.6078 - val_loss: 1.4619 - val_accuracy: 0.7255
Epoch 5/50
34/34 [==============================] - 1s 16ms/step - loss: 1.5166 - accuracy: 0.6152 - val_loss: 1.5079 - val_accuracy: 0.7255
Epoch 6/50
34/34 [==============================] - 1s 17ms/step - loss: 1.4684 - accuracy: 0.6250 - val_loss: 1.5100 - val_accuracy: 0.7255
Epoch 7/50
34/34 [==============================] - 1s 17ms/step - loss: 1.4433 - accuracy: 0.6225 - val_loss: 1.5978 - val_accuracy: 0.7255
Epoch 8/50
34

/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [ ]:
print("Shape of decoder_outputs:", decoder_outputs.shape)
print("Sample values of decoder_outputs:", decoder_outputs[0])

# Print shapes of input and target data
print("Shape of encoder_input_data:", encoder_input_data.shape)
print("Shape of decoder_input_data:", decoder_input_data.shape)
print("Shape of decoder_target_data:", decoder_target_data.shape)

# Print shapes of decoder outputs and states
print("Shape of decoder_outputs:", decoder_outputs.shape)
print("Shape of state_h:", state_h.shape)
print("Shape of state_c:", state_c.shape)

# Print sample values of decoder outputs and states
print("Sample values of decoder_outputs:", decoder_outputs[0])
print("Sample value of state_h:", state_h[0])
print("Sample value of state_c:", state_c[0])

Shape of decoder_outputs: (None, 6, 128)
Sample values of decoder_outputs: KerasTensor(type_spec=TensorSpec(shape=(6, 128), dtype=tf.float32, name=None), name='tf.__operators__.getitem_8/strided_slice:0', description="created by layer 'tf.__operators__.getitem_8'")
Shape of encoder_input_data: (85, 11)
Shape of decoder_input_data: (85, 6)
Shape of decoder_target_data: (85, 6)
Shape of decoder_outputs: (None, 6, 128)
Shape of state_h: (None, 128)
Shape of state_c: (None, 128)
Sample values of decoder_outputs: KerasTensor(type_spec=TensorSpec(shape=(6, 128), dtype=tf.float32, name=None), name='tf.__operators__.getitem_9/strided_slice:0', description="created by layer 'tf.__operators__.getitem_9'")
Sample value of state_h: KerasTensor(type_spec=TensorSpec(shape=(128,), dtype=tf.float32, name=None), name='tf.__operators__.getitem_10/strided_slice:0', description="created by layer 'tf.__operators__.getitem_10'")
Sample value of state_c: KerasTensor(type_spec=TensorSpec(shape=(128,), dtype=t

In [ ]:
user_input = input("Enter your input: ")
user_input_col = df.columns[0]
string_col = df.columns[1]
corresponding_string = df[df[user_input_col] == user_input][string_col].values

if len(corresponding_string) > 0:
    print("Original Sentence:", user_input)
    print("Gloss language:", corresponding_string[0])
else:
    sentence_without_stopwords = remove_stopwords(user_input)
    rearranged_example = rearrange_sentence(sentence_without_stopwords)
    print("Original Sentence:", user_input)
    print("Gloss Sentence:", rearranged_example)
    print()

Enter your input: how was your day
Original Sentence: how was your day
Gloss Sentence: day how was your

